# Import

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
import math

# Config

In [ ]:
CLS2_BLACKOUT = '/project/lt200264-saiwat/WormProject/data/release/worm-24022026-v3/views/cls2_blackout/'
CLS2_DIM = '/project/lt200264-saiwat/WormProject/data/release/worm-24022026-v3/views/cls2_dim/'
CLS2_RAW = '/project/lt200264-saiwat/WormProject/data/release/worm-24022026-v3/views/cls2_raw/'
CLS4_RAW = '/project/lt200264-saiwat/WormProject/data/release/worm-24022026-v3/views/cls4_raw/'
CLS5_RAW = '/project/lt200264-saiwat/WormProject/data/release/worm-24022026-v3/views/cls5_raw/'
MODEL_BENCHMARK_PATH = '/scratch/lt200264-saiwat/worm-project/worm-classification-model/model_benchmark/all_results.xlsx'
DATASET_BENCHMARK_PATH = '/scratch/lt200264-saiwat/worm-project/worm-classification-model/dataset_benchmark/all_results.xlsx'

# Helper function

In [ ]:
def plot_class_dist(dataset_dir):
    ds_path = Path(dataset_dir)
    splits = ['train', 'val', 'test']
    
    train_path = ds_path / 'train'
    if not train_path.exists():
        print(f"Train directory not found at {train_path}")
        return
        
    classes = sorted([d.name for d in train_path.iterdir() if d.is_dir()]) # get class folder name

    split_counts = {split: [] for split in splits}
    for split in splits:
        for cls in classes:
            cls_dir = ds_path / split / cls
            if cls_dir.exists():
                count = sum(1 for x in cls_dir.iterdir() if x.is_file()) # loop through each class folder
                split_counts[split].append(count)
            else:
                split_counts[split].append(0)
                
    x = np.arange(len(classes))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for i, split in enumerate(splits):
        offset = width * (i - 1)
        bars = ax.bar(x + offset, split_counts[split], width, label=split)
        ax.bar_label(bars, padding=3, fmt='%d')
        
    ax.set_xlabel('Classes')
    ax.set_ylabel('Number of Images')
    ax.set_title(f'Class Distribution in {ds_path.name}')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
def print_dataset_count(dataset_dir):
    ds_path = Path(dataset_dir)
    total_imgs = 0
    for split in ['train', 'val', 'test']:
        split_path = ds_path / split
        if split_path.exists():
            count = sum(1 for x in split_path.rglob('*') if x.is_file()) # find every thing inside path +1 if match
            print(f"{split.capitalize():<6}: {count} images")
            total_imgs += count
        else:
            print(f"{split.capitalize():<6}: 0 images (Folder not found)")
            
    print("-" * 25)
    print(f"Total : {total_imgs} images\n")

In [ ]:
def view_image_folder(dataset_dir, max_images=15):
    ds_path = Path(dataset_dir)
    
    # Collect image files (adjust extensions as needed for your dataset)
    valid_exts = {'.png', '.jpg', '.jpeg', '.bmp'}
    image_paths = [p for p in ds_path.rglob('*') if p.is_file() and p.suffix.lower() in valid_exts]
    
    # Limit images to prevent crashing the notebook with massive folders
    image_paths = image_paths[:max_images]
    num_images = len(image_paths)
    
    if num_images == 0:
        print(f"No images found in {dataset_dir}")
        return
        
    cols = 3
    rows = math.ceil(num_images / cols)
    
    # Create an n*3 grid
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    
    # Ensure axes is iterable even if there's only 1 row/col
    if isinstance(axes, np.ndarray):
        axes = axes.flatten()
    else:
        axes = [axes]
        
    for i, img_path in enumerate(image_paths):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(img_path.name)
        axes[i].axis('off')
        
    # Hide any remaining empty subplots if num_images isn't a multiple of 3
    for j in range(num_images, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

In [ ]:
def get_df_list(all_df_dict):
    df_list = []
    for sheet_name, df in all_df_dict.items():
        if 'class' not in sheet_name.lower():
            df_filtered = df[~df['dataset'].isin(['5class', '2class', '3class_stage_pad30'])]
            df_list.append(df_filtered)
            
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

In [ ]:
def plot_model_benchmark(all_df_dict, metric='val_f1_2class'):
    combined_df = get_df_list(all_df_dict)
    
    variant_avg = combined_df.groupby('variant')[metric].mean().sort_values(ascending=False)
    
    plt.figure(figsize=(24, 8))
    ax = sns.barplot(x=variant_avg.index, y=variant_avg.values) 
    
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', padding=3, fontsize=12) 
        
    plt.title(f'Average {metric} of Each Variant Across All Datasets', fontsize=14)
    plt.xlabel('Model Variant', fontsize=12)
    plt.ylabel(f'Mean {metric}', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
def get_top_models_from_excel(all_df_dict, metric='val_f1_2class', top_fams_n=3, top_variants_n=2):
    combined_df = get_df_list(all_df_dict)
    
    top_families = combined_df.groupby('family')[metric].mean().nlargest(top_fams_n).index.tolist()
    print(f"Top {top_fams_n} Families (Average): {top_families}")
    
    df_top_fams = combined_df[combined_df['family'].isin(top_families)]
    
    top_models = (
        df_top_fams.groupby(['family', 'variant'])[metric].mean().round(4)
        .reset_index()
        .sort_values(by=['family', metric], ascending=[True, False]) 
        .groupby('family')
        .head(top_variants_n) 
    ).reset_index(drop=True)
    
    return top_models

In [ ]:
def plot_dataset_benchmark(all_df_dict, metric='val_f1_2class'):
    combined_df = get_df_list(all_df_dict)
    
    dataset_avg = combined_df.groupby('dataset')[metric].mean().sort_values(ascending=False)
    
    plt.figure(figsize=(24, 8))
    
    ax = sns.barplot(x=dataset_avg.index, y=dataset_avg.values) 
    
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', padding=3, fontsize=12) 
        
    plt.title(f'Average {metric} of Each Dataset Across All Models', fontsize=14)
    plt.xlabel('Dataset', fontsize=12)
    plt.ylabel(f'Mean {metric}', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.tight_layout()
    plt.show()

# Model benchmark

In [ ]:
model_df = pd.read_excel(MODEL_BENCHMARK_PATH, sheet_name=None)
plot_model_benchmark(model_df)

# Select top 3 best model

In [ ]:
top_models_to_use = get_top_models_from_excel(model_df)

top_models_to_use[['family', 'variant', 'val_f1_2class']]

# Dataset benchmark

In [ ]:
dataset_df = pd.read_excel(DATASET_BENCHMARK_PATH, sheet_name=None)
plot_dataset_benchmark(dataset_df)

# 2 class blackout

In [ ]:
plot_class_dist(CLS2_BLACKOUT)
print_dataset_count(CLS2_BLACKOUT)

In [ ]:
view_image_folder(CLS2_BLACKOUT)

# 2 class dim

In [ ]:
plot_class_dist(CLS2_DIM)
print_dataset_count(CLS2_DIM)

In [ ]:
view_image_folder(CLS2_DIM)

# 2 class

In [ ]:
plot_class_dist(CLS2_RAW)
print_dataset_count(CLS2_RAW)

In [ ]:
view_image_folder(CLS2_RAW)

# 4 class

In [ ]:
plot_class_dist(CLS4_RAW)
print_dataset_count(CLS4_RAW)

In [ ]:
view_image_folder(CLS4_RAW)

# 5 class

In [ ]:
plot_class_dist(CLS5_RAW)
print_dataset_count(CLS5_RAW)

In [ ]:
view_image_folder(CLS5_RAW)